In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [2]:
import openfold
from openfold.utils.kernel import attention_core
print(attention_core)

<module 'openfold.utils.kernel.attention_core' from '/home/ubuntu/miniforge3/envs/esmfold_env/lib/python3.9/site-packages/openfold/utils/kernel/attention_core.py'>


In [16]:
#@title ##run **ESMFold**
from string import ascii_uppercase, ascii_lowercase
import hashlib, re, os
import numpy as np
import torch
from jax.tree_util import tree_map
import matplotlib.pyplot as plt
from scipy.special import softmax
import gc
import os, time

version = "1" # @param ["0", "1"]
model_name = "esmfold_v0.model" if version == "0" else "esmfold.model"

def parse_output(output):
  pae = (output["aligned_confidence_probs"][0] * np.arange(64)).mean(-1) * 31
  plddt = output["plddt"][0,:,1]

  bins = np.append(0,np.linspace(2.3125,21.6875,63))
  sm_contacts = softmax(output["distogram_logits"],-1)[0]
  sm_contacts = sm_contacts[...,bins<8].sum(-1)
  xyz = output["positions"][-1,0,:,1]
  mask = output["atom37_atom_exists"][0,:,1] == 1
  o = {"pae":pae[mask,:][:,mask],
       "plddt":plddt[mask],
       "sm_contacts":sm_contacts[mask,:][:,mask],
       "xyz":xyz[mask]}
  return o

def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
alphabet_list = list(ascii_uppercase+ascii_lowercase)

jobname = "1AGY" #@param {type:"string"}
jobname = re.sub(r'\W+', '', jobname)[:50]

def get_circular_permutations(seq):
  """Generate all circular permutations of a sequence with their cut positions."""
  return [(i, seq[i:] + seq[:i]) for i in range(len(seq))]

sequence = "LGRTTRDDLINGNSASCRDVIFIYARGSTETGNLGTLGPSIASNLESAFGKDGVWIQGVGGAYRATLGDNALPRGTSSAAIREMLGLFQQANTKCPDATLIAGGYSQGAALAAASIEDLDSAIRDKIAGTVLFGYTKNLQNRGRIPNYPADRTKVFCNTGDLVCTGSLIVAAPHLAYGPDARGPAPEFLIEKVRAVRGSA" #@param {type:"string"}
sequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())
sequence = re.sub(":+",":",sequence)
sequence = re.sub("^[:]+","",sequence)
sequence = re.sub("[:]+$","",sequence)
copies = 1 #@param {type:"integer"}
if copies == "" or copies <= 0: copies = 1
sequence = ":".join([sequence] * copies)
num_recycles = 3 #@param ["0", "1", "2", "3", "6", "12", "24"] {type:"raw"}
chain_linker = 25
# Use only the first chain for circular permutations
base_sequence = sequence.split(":")[0]
circular_permutations = get_circular_permutations(base_sequence)
print(f"Generating {len(circular_permutations)} circular permutations of length {len(base_sequence)}")


ID = jobname+"_"+get_hash(sequence)[:5]
seqs = sequence.split(":")
lengths = [len(s) for s in seqs]
length = sum(lengths)
print("length",length)

u_seqs = list(set(seqs))
if len(seqs) == 1: mode = "mono"
elif len(u_seqs) == 1: mode = "homo"
else: mode = "hetero"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = torch.load(model_name, weights_only=False)
model = model.eval().to(device)
model.requires_grad_(False)

model_name_ = model_name

os.system(f"mkdir -p {jobname}_CP")

# --- Main loop over all circular permutations ---
for cp_idx, (cut_pos, cp_sequence) in enumerate(circular_permutations):
  print(f"\n[{cp_idx+1}/{len(circular_permutations)}] cut_pos={cut_pos}")

  ID = f"{jobname}_CP{cut_pos}_" + get_hash(cp_sequence)[:5]
  length = len(cp_sequence)

  # optimized for Tesla T4
  if length > 700:
    model.set_chunk_size(64)
  else:
    model.set_chunk_size(128)

  torch.cuda.empty_cache()
  try:
    output = model.infer(cp_sequence,
                         num_recycles=num_recycles,
                         chain_linker="X"*chain_linker,
                         residue_index_offset=512)

    pdb_str = model.output_to_pdb(output)[0]
    output = tree_map(lambda x: x.cpu().numpy(), output)
    ptm = output["ptm"][0]
    plddt = output["plddt"][0,...,1].mean()
    O = parse_output(output)
    print(f'  ptm: {ptm:.3f} plddt: {plddt:.3f}')

    prefix = f"{jobname}_CP/cut{cut_pos:04d}_ptm{ptm:.3f}_r{num_recycles}"
    np.savetxt(f"{prefix}.pae.txt", O["pae"], "%.3f")
    np.savetxt(f"{prefix}.plddt.txt", O["plddt"], "%.3f")
    with open(f"{prefix}.pdb", "w") as out:
      out.write(pdb_str)

  except Exception as e:
    print(f"  ERROR at cut_pos={cut_pos}: {e}")
    continue

print("\nDone! All circular permutations complete.")

Generating 200 circular permutations of length 200
length 200
cuda

[1/200] cut_pos=0
  ptm: 0.933 plddt: 93.675

[2/200] cut_pos=1
  ptm: 0.936 plddt: 94.168

[3/200] cut_pos=2
  ptm: 0.934 plddt: 94.011

[4/200] cut_pos=3
  ptm: 0.929 plddt: 93.432

[5/200] cut_pos=4
  ptm: 0.927 plddt: 93.060

[6/200] cut_pos=5
  ptm: 0.922 plddt: 92.266

[7/200] cut_pos=6
  ptm: 0.919 plddt: 92.260

[8/200] cut_pos=7
  ptm: 0.917 plddt: 92.174

[9/200] cut_pos=8
  ptm: 0.935 plddt: 94.028

[10/200] cut_pos=9
  ptm: 0.923 plddt: 92.772

[11/200] cut_pos=10
  ptm: 0.897 plddt: 90.222

[12/200] cut_pos=11
  ptm: 0.898 plddt: 90.509

[13/200] cut_pos=12
  ptm: 0.901 plddt: 91.210

[14/200] cut_pos=13
  ptm: 0.900 plddt: 91.408

[15/200] cut_pos=14
  ptm: 0.899 plddt: 91.517

[16/200] cut_pos=15
  ptm: 0.898 plddt: 91.657

[17/200] cut_pos=16
  ptm: 0.931 plddt: 95.412

[18/200] cut_pos=17
  ptm: 0.887 plddt: 89.776

[19/200] cut_pos=18
  ptm: 0.880 plddt: 88.355

[20/200] cut_pos=19
  ptm: 0.872 plddt:

In [11]:
output["plddt"]

array([[[37.953426, 40.11216 , 39.00833 , ..., 51.744728, 30.896204,
         47.813942],
        [48.112923, 48.13139 , 48.147438, ..., 56.000687, 32.566082,
         49.015827],
        [66.888275, 66.494576, 68.486664, ..., 62.166656, 39.079777,
         51.301533],
        ...,
        [93.14788 , 92.968666, 91.60411 , ..., 76.61728 , 76.786026,
         59.83466 ],
        [90.511826, 89.76706 , 88.35098 , ..., 74.63353 , 75.33088 ,
         58.948654],
        [87.82118 , 86.65058 , 83.878784, ..., 74.71424 , 73.659225,
         58.36453 ]]], dtype=float32)

In [ ]:
output

In [8]:
import py3Dmol
pymol_color_list = ["#33ff33","#00ffff","#ff33cc","#ffff00","#ff9999","#e5e5e5","#7f7fff","#ff7f00",
                    "#7fff7f","#199999","#ff007f","#ffdd5e","#8c3f99","#b2b2b2","#007fff","#c4b200",
                    "#8cb266","#00bfbf","#b27f7f","#fcd1a5","#ff7f7f","#ffbfdd","#7fffff","#ffff7f",
                    "#00ff7f","#337fcc","#d8337f","#bfff3f","#ff7fff","#d8d8ff","#3fffbf","#b78c4c",
                    "#339933","#66b2b2","#ba8c84","#84bf00","#b24c66","#7f7f7f","#3f3fa5","#a5512b"]

def show_pdb(pdb_str, show_sidechains=False, show_mainchains=False,
             color="pLDDT", chains=None, vmin=50, vmax=90,
             size=(800,480), hbondCutoff=4.0,
             Ls=None,
             animate=False):

  if chains is None:
    chains = 1 if Ls is None else len(Ls)
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=size[0], height=size[1])
  if animate:
    view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  else:
    view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  if color == "pLDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':vmin,'max':vmax}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                  {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  view.zoomTo()
  if animate: view.animate()
  return view

color = "confidence" #@param ["confidence", "rainbow", "chain"]
if color == "confidence": color = "pLDDT"
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
show_pdb(pdb_str, color=color,
         show_sidechains=show_sidechains,
         show_mainchains=show_mainchains,
         Ls=lengths).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
import torch
import esm

In [ ]:
model = esm.pretrained.esmfold_v1()

In [ ]:
model = model.eval().cuda()

In [ ]:
# Optionally, uncomment to set a chunk size for axial attention. This can help reduce memory.
# Lower sizes will have lower memory requirements at the cost of increased speed.
# model.set_chunk_size(128)

sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
# Multimer prediction can be done with chains separated by ':'

with torch.no_grad():
    output = model.infer_pdb(sequence)

with open("result.pdb", "w") as f:
    f.write(output)

import biotite.structure.io as bsio
struct = bsio.load_structure("result.pdb", extra_fields=["b_factor"])
print(struct.b_factor.mean())  # this will be the pLDDT
# 88.3